# Decomposing Private Equity

**FINM 36700 · Homework 3 · Group 22**  
Lucas Albín Lazcano, Avi Rastogi, Mengting Wang, and Yihan Kong

*Acknowledgements: ChatGPT 5.6 was used to help explain the concepts, organize the submission, and assist with coding the solutions.*

## Data

This problem uses the weekly excess returns in `private_equity_data.xlsx`.

In [ ]:
import pandas as pd
import numpy as np

excess_returns = pd.read_excel(
    "private_equity_data.xlsx",
    sheet_name="excess returns",
    index_col="date",
    parse_dates=True,
)
excess_returns.head()

# 1. Performance

Report the performance statistics of the assets.

The stand-alone measures are

- mean
- volatility
- Sharpe ratio

The decomposition measures, relative to SPY, are

- alpha
- beta
- information ratio
- $R^2$

In [ ]:
standalone = pd.DataFrame({
    "Annualized mean": excess_returns.mean() * 52,
    "Annualized volatility": excess_returns.std() * np.sqrt(52),
})
standalone["Sharpe ratio"] = (
    standalone["Annualized mean"] / standalone["Annualized volatility"]
)

standalone.style.format({
    "Annualized mean": "{:.2%}",
    "Annualized volatility": "{:.2%}",
    "Sharpe ratio": "{:.3f}",
})

In [ ]:
def fit_ols(y, x):
    """Estimate an OLS regression with an intercept using NumPy."""
    x = pd.DataFrame(x).copy()
    design = pd.concat(
        [pd.Series(1.0, index=x.index, name="const"), x], axis=1
    )
    coefficients = np.linalg.lstsq(
        design.to_numpy(), y.to_numpy(), rcond=None
    )[0]
    params = pd.Series(coefficients, index=design.columns)
    residuals = y - design @ params
    r_squared = 1 - (residuals ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return params, residuals, r_squared


def regression_decomposition(y, market):
    params, residuals, r_squared = fit_ols(y, market)
    alpha = params["const"] * 52
    tracking_error = residuals.std() * np.sqrt(52)
    return pd.Series({
        "Annualized alpha": alpha,
        "Beta": params[market.name],
        "Information ratio": alpha / tracking_error,
        "R-squared": r_squared,
    })


decomposition = pd.DataFrame({
    asset: regression_decomposition(excess_returns[asset], excess_returns["SPY"])
    for asset in excess_returns.columns
    if asset != "SPY"
}).T

decomposition.style.format({
    "Annualized alpha": "{:.2%}",
    "Beta": "{:.3f}",
    "Information ratio": "{:.3f}",
    "R-squared": "{:.3f}",
})

_Answer:_ SPY earns an annualized mean excess return of 12.31% with 16.97% volatility, producing the highest stand-alone Sharpe ratio in the table at 0.726. Among the listed private-equity firms, APO has the highest mean excess return at 23.15%, while APO, BX, and KKR all have Sharpe ratios close to 0.6. These firms also have much greater total risk than SPY, with annualized volatility around 36%–37%.

The regression results show that the public private-equity firms carry substantial equity-market exposure. Their SPY betas range from about 1.46 to 1.57, so their returns tend to move more than the broad market. SPY alone explains 46%–54% of their weekly variation. APO has the largest positive annualized alpha at 4.93%, followed by BX at 3.74% and KKR at 3.48%, but their information ratios are modest because the residual volatility is large. CG has a negative alpha of -3.59%. The ETF results are weaker: PSP and PEX both have negative alpha relative to SPY, despite their high market exposure.

# 2. Replication

Can we replicate APO using SPY, IEF, and HYG?

How does this compare to the decomposition of APO against SPY alone?

In [ ]:
factors = ["SPY", "IEF", "HYG"]
replication_params, replication_residuals, replication_r_squared = fit_ols(
    excess_returns["APO"], excess_returns[factors]
)

annualized_alpha = replication_params["const"] * 52
tracking_error = replication_residuals.std() * np.sqrt(52)

replication_summary = pd.Series({
    "Annualized alpha": annualized_alpha,
    "SPY beta": replication_params["SPY"],
    "IEF beta": replication_params["IEF"],
    "HYG beta": replication_params["HYG"],
    "Tracking error": tracking_error,
    "Information ratio": annualized_alpha / tracking_error,
    "R-squared": replication_r_squared,
}, name="SPY + IEF + HYG")

replication_summary.to_frame().style.format("{:.3f}")

In [ ]:
# A tradable factor-only replication excludes the regression intercept.
replication_return = excess_returns[factors] @ replication_params[factors]

apo_vs_replication = pd.DataFrame({
    "Annualized mean": pd.concat(
        [excess_returns["APO"], replication_return], axis=1
    ).mean() * 52,
    "Annualized volatility": pd.concat(
        [excess_returns["APO"], replication_return], axis=1
    ).std() * np.sqrt(52),
})
apo_vs_replication.index = ["APO", "Factor-only replication"]
apo_vs_replication["Sharpe ratio"] = (
    apo_vs_replication["Annualized mean"]
    / apo_vs_replication["Annualized volatility"]
)

apo_vs_replication.style.format({
    "Annualized mean": "{:.2%}",
    "Annualized volatility": "{:.2%}",
    "Sharpe ratio": "{:.3f}",
})

_Answer:_ The estimated factor replication uses weights of 1.066 in SPY, -0.704 in IEF, and 0.931 in HYG. This combination captures APO's broad equity and credit exposures, but it is not a complete replication. The model has an $R^2$ of 0.497, meaning that the three factors explain about half of APO's weekly return variation. The remaining annualized tracking error is still 26.11%.

The tradable factor-only portfolio has an annualized mean excess return of 13.19%, volatility of 25.92%, and a Sharpe ratio of 0.509. APO itself earns 23.15% with 36.79% volatility and a Sharpe ratio of 0.629. Therefore, the factors reproduce part of APO's systematic risk, but they do not reproduce its full average return or its firm-specific risk. The positive annualized intercept of 9.96% represents the portion of APO's average return that is not delivered by the factor portfolio.

In [ ]:
spy_params, spy_residuals, spy_r_squared = fit_ols(
    excess_returns["APO"], excess_returns[["SPY"]]
)


def model_statistics(params, residuals, r_squared):
    alpha = params["const"] * 52
    tracking_error = residuals.std() * np.sqrt(52)
    return pd.Series({
        "Annualized alpha": alpha,
        "SPY beta": params["SPY"],
        "Tracking error": tracking_error,
        "Information ratio": alpha / tracking_error,
        "R-squared": r_squared,
    })


model_comparison = pd.DataFrame({
    "SPY only": model_statistics(
        spy_params, spy_residuals, spy_r_squared
    ),
    "SPY + IEF + HYG": model_statistics(
        replication_params, replication_residuals, replication_r_squared
    ),
}).T

model_comparison.style.format({
    "Annualized alpha": "{:.2%}",
    "SPY beta": "{:.3f}",
    "Tracking error": "{:.2%}",
    "Information ratio": "{:.3f}",
    "R-squared": "{:.3f}",
})

_Answer:_ Adding IEF and HYG raises the $R^2$ from 0.466 in the SPY-only model to 0.497 and lowers annualized tracking error from 26.88% to 26.11%. The improvement is real but small, so SPY remains the main source of explanatory power. The SPY beta falls from 1.480 to 1.066 after the bond factors are included because HYG captures part of the credit-sensitive exposure that the one-factor regression had attributed to SPY. The estimated alpha rises from 4.93% to 9.96%, showing that alpha depends on the benchmark used. Overall, SPY, IEF, and HYG provide a somewhat better decomposition than SPY alone, but neither model closely replicates APO.